# Phase 3 & 4 — Stage 1: QLoRA SFT Training Loop
### Llama-3.2-3B SFT & Alignment Project

**Goal**: Train the raw `Llama-3.2-3B` base model using **QLoRA (4-bit NF4)** on our curated Stage 1 SFT dataset (`OpenHermes-2.5`, `Dolly-15k`, `GSM8K`, `MetaMathQA`, `HotpotQA`).

**What you will learn & execute:**
1. How **QLoRA** quantizes base weights into 4-bit NF4 while keeping trainable LoRA adapters in 16-bit.
2. How **Unsloth** accelerates training by 2-5x and cuts VRAM usage by 60% with custom CUDA kernels.
3. How to load your private dataset directly from HuggingFace Hub.
4. How to configure `SFTTrainer` with gradient accumulation, cosine LR decay, and grad norm clipping.
5. How to evaluate qualitative responses before vs. after training.
6. How to save adapters and push the Stage 1 model to HuggingFace Hub.

---
### Architecture Choice: Why Unsloth over Raw HuggingFace?

- **2-5x faster training** via custom hand-written CUDA kernels for attention and cross-entropy
- **~60% VRAM reduction** — makes T4 (15GB) a viable training target
- **Gradient checkpointing in `"unsloth"` mode** — more memory-efficient than the vanilla HF version
- **Drop-in compatibility** with `SFTTrainer`, `LoraConfig`, and HF Hub — no workflow change



## 1. Setup & Environment Check

In [1]:
import warnings
warnings.filterwarnings('ignore') # Ignore warnings from unsloth_zoo for now

# Install Unsloth with Colab-optimized builds and all required dependencies
# --no-deps prevents pip from pulling conflicting transitive dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers huggingface_hub datasets unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-u9guy__t/unsloth_eadea1d6b658476682daa125e2fa9091
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-u9guy__t/unsloth_eadea1d6b658476682daa125e2fa9091
  Resolved https://github.com/unslothai/unsloth.git to commit 1838e1084c481e6a26acf902ca772d36a2bfcf06
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.8.16-py3-none-any.whl size=46131442 sha256=82db6b01b832fb89667415364f5ee098f7604f744870ef4f3f7e0ad01db7fcd2
  Stored in directory: /tmp/pip-ephem-wheel-cache-e3p_i7p4/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 71.4 MB/s eta 0:00:00
   ━━━

In [2]:
# Verify GPU is available and get hardware specs
# This matters because:
#   - T4 (15GB): use per_device_batch_size=4 + grad_accum=8 to hit effective batch_size=32
#   - A100 (40/80GB): can use per_device_batch_size=32 directly
#   - bf16 support: A100/L4 support bfloat16; T4 only supports fp16
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM Available  : {total_vram:.2f} GB")
    print(f"bf16 Supported  : {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → GPU.")

PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU Device Name : Tesla T4
VRAM Available  : 15.64 GB
bf16 Supported  : True


In [3]:
# Authenticate with HuggingFace Hub
# Required to:
#   1. Load our private SFT dataset (from Phase 2 pipeline)
#   2. Push our trained adapter back to Hub after training
from huggingface_hub import notebook_login
notebook_login()

## 2. Configuration & Hyperparameters

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — single source of truth for all hyperparameters
# Change values HERE only — every subsequent cell reads from these variables
# ─────────────────────────────────────────────────────────────────────────────

# ── HuggingFace Identity ──────────────────────────────────────────────────────
HF_USERNAME       = "SpongeBob387"                          # <-- Your HF username
SFT_DATASET_REPO  = f"{HF_USERNAME}/llama3.2-3b-sft-dataset" # Phase 2 output repo
STAGE1_MODEL_REPO = f"{HF_USERNAME}/llama3.2-3b-sft-stage1" # Where adapter will be pushed

# ── Model Identity ────────────────────────────────────────────────────────────
# We use unsloth's pre-patched version of Llama-3.2-3B.
# 'unsloth/Llama-3.2-3B' = the same base weights as 'meta-llama/Llama-3.2-3B'
# but pre-patched with Unsloth's CUDA kernels. No quality difference.
MODEL_NAME     = "unsloth/Llama-3.2-3B"
MAX_SEQ_LENGTH = 512   # Token sequence limit. Locked in Phase 1 (O(n²) attention cost)
LOAD_IN_4BIT   = True  # 4-bit NF4 quantization. Reduces ~6GB model → ~1.8GB GPU footprint

# ── LoRA Adapter Configuration ───────────────────────────────────────────────
# Reference (Tan's notebook): r=64, alpha=16, dropout=0.1
# Our choice: r=16, alpha=16, dropout=0 — because:
#   - r=64 requires ~8GB extra VRAM for the adapter matrices. OOM risk on T4.
#   - r=16 is the standard T4-safe LoRA rank. Still injects 40M+ trainable params.
#   - dropout=0 is required by Unsloth: its custom kernels are incompatible with
#     non-zero LoRA dropout (using it produces an immediate warning and fallback).
LORA_R         = 16
LORA_ALPHA     = 16    # Scaling factor. Standard practice: alpha = r or 2*r
LORA_DROPOUT   = 0     # Must be 0 when using Unsloth (see rationale above)

# All 7 projection layers targeted — maximum expressiveness for the LoRA rank budget.
# q_proj, k_proj, v_proj, o_proj = attention mechanism
# gate_proj, up_proj, down_proj  = MLP (feed-forward) layers
# Tan's notebook uses the same 7 target modules.
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training Hyperparameters ─────────────────────────────────────────────────
# Effective batch size = PER_DEVICE_BATCH_SIZE × GRAD_ACCUM_STEPS = 4 × 8 = 32
# This matches Tan's batch_size=32 (which he ran directly on A100 with no accumulation).
# Gradient accumulation simulates a larger batch by accumulating gradients over N
# micro-steps before performing one weight update — same math, different memory profile.
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS      = 8
LEARNING_RATE         = 2e-4    # Standard QLoRA SFT learning rate
NUM_EPOCHS            = 1       # Reference trained 1 epoch on 88k samples
WARMUP_RATIO          = 0.03    # First 3% of steps ramp LR from 0 → LEARNING_RATE
WEIGHT_DECAY          = 0.001   # L2 regularization (aligned with Tan's 0.001)

# max_grad_norm: clip gradients whose L2 norm exceeds this value.
# Prevents exploding gradients, especially important in QLoRA where base weights
# are frozen at 4-bit precision and adapter gradients can spike.
# Tan uses 0.3. Standard for QLoRA SFT.
MAX_GRAD_NORM         = 0.3

print(f"Config ready.")
print(f"  Dataset  : {SFT_DATASET_REPO}")
print(f"  Push to  : {STAGE1_MODEL_REPO}")
print(f"  Eff. Batch: {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS}")

Config ready.
  Dataset  : SpongeBob387/llama3.2-3b-sft-dataset
  Push to  : SpongeBob387/llama3.2-3b-sft-stage1
  Eff. Batch: 32


## 3. Load Base Model & Attach LoRA Adapters via Unsloth

In [5]:
from unsloth import FastLanguageModel

# Step 1: Load the base model in 4-bit NF4 quantization
# NF4 (Normal Float 4-bit) is optimal for normally distributed neural network weights.
# Unsloth handles the BitsAndBytesConfig internally — it patches the model with its
# own CUDA kernels for NF4 dequantization AND forward pass, giving the 2-5x speedup.
# dtype=None → Unsloth auto-detects: bfloat16 on A100/L4, float16 on T4
print("Loading base Llama-3.2-3B with Unsloth in 4-bit QLoRA...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect fp16/bf16 based on GPU capability
    load_in_4bit=LOAD_IN_4BIT,
)

# Step 2: Inject LoRA adapters into the target projection layers
# After this call:
#   - Base model weights: frozen, 4-bit NF4, not updated during training
#   - LoRA adapter matrices (A and B for each target layer): trainable, fp16/bf16
#   - Trainable params ≈ 40M out of 3.3B total (only ~2.9% of all params are trained)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # 'unsloth' gradient checkpointing: Unsloth's custom implementation that
    # uses less VRAM than vanilla HF gradient checkpointing by keeping fewer
    # intermediate activations in GPU memory during the backward pass.
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("\n✅ QLoRA Model Initialized!")
model.print_trainable_parameters()

# Also ensure pad_token is set — Llama base model has no pad token by default.
# We set it equal to eos_token (same as reference), which is standard for causal LM SFT.
# Right-side padding ensures the completion tokens are not masked during training.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Loading base Llama-3.2-3B with Unsloth in 4-bit QLoRA...
==((====))==  Unsloth 2026.8.16: Fast Llama patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.16 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✅ QLoRA Model Initialized!
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
Pad token: <|finetune_right_pad_id|> (id=128004)


## 4. Load SFT Dataset from HuggingFace Hub

In [6]:
from datasets import load_dataset
from huggingface_hub import whoami

try:
    user_info = whoami()
    print(f"Logged in as: {user_info['name']}")
except:
    print("Warning: Not logged into HuggingFace. Private datasets will not be accessible.")

# Our Phase 2 pipeline pushed a pre-formatted dataset to HuggingFace Hub.
print(f"Downloading SFT dataset: {SFT_DATASET_REPO}")

try:
    sft_dataset = load_dataset(SFT_DATASET_REPO)

    train_data = sft_dataset["train"]
    eval_data  = sft_dataset["test"]

    print(f"Train samples : {len(train_data):,}")
    print(f"Eval samples  : {len(eval_data):,}")
    print("\nSample preview (first 300 chars):")
    print("-" * 60)
    print(train_data[0]["text"][:300])
    print("-" * 60)
except Exception as e:
    print(f"\n❌ ERROR: Could not load dataset '{SFT_DATASET_REPO}'.")
    print("Checklist:")
    print("1. Does the repo exist at https://huggingface.co/datasets/" + SFT_DATASET_REPO + "?")
    print("2. If private, did you use a Write/Read token in the login step?")
    print(f"\nFull error: {e}")

Logged in as: SpongeBob387


README.md:   0%|          | 0.00/397 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.45MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.9kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10974 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/224 [00:00<?, ? examples/s]

Train samples : 10,974
Eval samples  : 224

Sample preview (first 300 chars):
------------------------------------------------------------
Instruction:
No. 24 Louisville 34, North Carolina 0

Eric Shelton ran for three touchdowns and No. 24 Louisville earned its second shutout of the season with a 34-0 victory over North Carolina on Saturday.

What is this text about?

### Response:
Step 1: Identify the general topic - The text seems t
------------------------------------------------------------


In [7]:
import unsloth.models.llama as _llama_mod
import unsloth_zoo.fused_losses.cross_entropy_loss as _fused_ce_mod
import torch
import torch.nn.functional as F

def _standard_ce_loss(trainer, hidden_states, lm_head_weight, lm_head_bias=None,
                      labels=None, mask=None, n_items=None, scaling=1.0,
                      target_gb=1, torch_compile=False, overwrite=True,
                      shift_labels=True, **kwargs):
    """
    Standard CE — replaces Unsloth's buggy chunked CE.
    Handles the off-by-one where SFT labels are 1 token longer than hidden_states.
    """
    logits = torch.matmul(hidden_states.float(), lm_head_weight.float().T)
    if lm_head_bias is not None:
        logits = logits + lm_head_bias.float()

    if shift_labels:
        logits = logits[..., :-1, :].contiguous()   # (B, T-1, V)
        labels = labels[..., 1:].contiguous()        # (B, T'-1)

    # Align to the shorter dimension — handles the off-by-one from SFTTrainer's
    # tokenizer adding EOS, making labels 1 token longer than hidden_states.
    seq_len = min(logits.size(1), labels.size(1))
    logits = logits[:, :seq_len, :]
    labels = labels[:, :seq_len]

    vocab_size = logits.size(-1)
    reduction = "sum" if n_items is not None else "mean"

    loss = F.cross_entropy(
        logits.view(-1, vocab_size),
        labels.view(-1).long(),
        ignore_index=-100,
        reduction=reduction
    )

    if n_items is not None:
        loss = loss / n_items

    return loss

_fused_ce_mod.unsloth_fused_ce_loss = _standard_ce_loss
_llama_mod.unsloth_fused_ce_loss = _standard_ce_loss

print("✅ Patch v4 applied (min-size alignment). Recreate trainer and run.")

✅ Patch v4 applied (min-size alignment). Recreate trainer and run.


## 5. Configure & Execute SFT Training Loop

In [10]:
from trl import SFTTrainer, SFTConfig
import os
import torch

# Create output directories
OUTPUT_DIR  = "./sft_stage1_output"
LOGGING_DIR = "./sft_stage1_logs"
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

# Calculate warmup steps
total_steps = (len(train_data) // (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="paged_adamw_32bit",
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=10,
        report_to="none",
        seed=42,
    ),
)

# Disable Unsloth's fused loss to fix potential TorchRuntimeErrors (shape mismatches)
trainer.model.config.use_fused_cross_entropy = False

print(f"Starting Stage 1 QLoRA SFT Training (Warmup steps: {warmup_steps})...")
trainer_stats = trainer.train()

print(f"\n✅ Stage 1 Training Complete!")
print(f"   Runtime   : {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"   Train Loss: {trainer_stats.metrics['train_loss']:.4f}")

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting Stage 1 QLoRA SFT Training (Warmup steps: 10)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,974 | Num Epochs = 1 | Total steps = 343
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,Validation Loss
200,1.082618,1.146157
343,1.133404,1.136685


Unsloth: Restored added_tokens_decoder metadata in ./sft_stage1_output/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_stage1_output/checkpoint-343/tokenizer_config.json.



✅ Stage 1 Training Complete!
   Runtime   : 3889.08 seconds
   Train Loss: 1.1717


## 6. Qualitative Sanity Check (Test Inference)

In [11]:
# Switch model from training mode to optimized inference mode.
# FastLanguageModel.for_inference() disables dropout layers, enables KV-cache,
# and activates Unsloth's 2x faster inference kernels.
FastLanguageModel.for_inference(model)

# Test prompts use the SAME Alpaca format as training:
#   Instruction:\n{text}\n\n### Response:\n
# This is critical — if the format at inference doesn't match training, the model
# won't know when its response begins and will produce incoherent outputs.
test_prompts = [
    "Instruction:\nExplain the difference between supervised and unsupervised learning.\n\n### Response:\n",
    "Instruction:\nSolve the following math problem step by step.\nIf a train travels at 60 mph for 2.5 hours, how far does it travel?\n\n### Response:\n",
    "Instruction:\nWhat is the capital of France, and what is it known for?\n\n### Response:\n",
]

for i, prompt in enumerate(test_prompts):
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,   # KV-cache: reuses previously computed key/value pairs → faster
    )
    # Decode only the generated tokens (not the prompt)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"=== Test Prompt {i+1} ===")
    print(f"Prompt: {prompt.strip()[:80]}...")
    print(f"Response: {response}")
    print("-" * 60)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test Prompt 1 ===
Prompt: Instruction:
Explain the difference between supervised and unsupervised learning...
Response: Supervised learning is a machine learning technique in which a computer system is given a set of input data and is expected to produce an output label for each data point. The output labels are what the machine learning algorithm is trying to learn from. For example, supervised learning might be used to train a computer system to recognize handwritten digits, or to classify emails as spam or non-spam. In supervised learning, the computer system is given both the input data and the corresponding output labels. The computer system then uses this information to learn how to predict the output labels for new input data that it has not seen before.

Unsupervised learning is a machine learning technique in which a computer system is given a set of input data, but is not expected to produce an output label for each data point. Instead, the computer system is expected to 

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test Prompt 2 ===
Prompt: Instruction:
Solve the following math problem step by step.
If a train travels a...
Response: The train travels 60 mph for 2.5 hours, so it travels 60 x 2.5 = <<60*2.5=150>>150 miles.
#### 150
------------------------------------------------------------
=== Test Prompt 3 ===
Prompt: Instruction:
What is the capital of France, and what is it known for?

### Respo...
Response: Paris is the capital of France, and is known for its rich history, architecture, and food. It is a major center of art and culture, and is home to many famous landmarks such as the Eiffel Tower and the Louvre.
------------------------------------------------------------


## 7. Save Adapter & Push Model to HuggingFace Hub

In [12]:
# Save the LoRA adapter weights (NOT the full model — just the ~40M trainable params).
# The adapter directory will contain:
#   - adapter_config.json  ← LoRA architecture config (r, alpha, target_modules, etc.)
#   - adapter_model.bin    ← The actual trained adapter weights
#   - tokenizer files      ← Needed for loading later
#
# Why save adapters (not full merged model)?
#   - Adapter: ~160MB vs. merged model: ~6GB
#   - We can merge later (Phase 4.5) before deploying with vLLM
#   - Easier to iterate and resume training from an adapter checkpoint
LOCAL_SAVE_DIR = "./llama3.2-3b-sft-stage1-adapter"
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)
print(f"✅ Adapter saved locally to: {LOCAL_SAVE_DIR}")

# Push to HuggingFace Hub (private repo)
print(f"\nPushing Stage 1 SFT Adapter to: {STAGE1_MODEL_REPO}")
model.push_to_hub(STAGE1_MODEL_REPO, private=True)
tokenizer.push_to_hub(STAGE1_MODEL_REPO, private=True)
print(f"\n✅ Stage 1 SFT Adapter pushed to: https://huggingface.co/{STAGE1_MODEL_REPO}")

Unsloth: Restored added_tokens_decoder metadata in ./llama3.2-3b-sft-stage1-adapter/tokenizer_config.json.


✅ Adapter saved locally to: ./llama3.2-3b-sft-stage1-adapter

Pushing Stage 1 SFT Adapter to: SpongeBob387/llama3.2-3b-sft-stage1


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

... (2 outputs left out)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpsnltuwlc/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            


✅ Stage 1 SFT Adapter pushed to: https://huggingface.co/SpongeBob387/llama3.2-3b-sft-stage1


## Phase 3 & 4 Complete

| Deliverable | Location | Status |
|---|---|---|
| **Stage 1 SFT Adapter** | `https://huggingface.co/{STAGE1_MODEL_REPO}` | ✅ Saved & Pushed |

### Training Config Summary

| Parameter | Value | Note |
|-----------|-------|------|
| `lora_r` | 16 | T4-safe (Tan used r=64 on A100) |
| `lora_alpha` | 16 | Standard scaling |
| `lora_dropout` | 0 | Required by Unsloth |
| `target_modules` | q,k,v,o,gate,up,down | Matches reference |
| `learning_rate` | 2e-4 | Matches reference |
| `effective_batch_size` | 32 | 4 × 8 grad accum |
| `max_grad_norm` | 0.3 | Matches reference |
| `group_by_length` | True | Matches reference |
| `num_train_epochs` | 1 | Matches reference |
| `lr_scheduler` | cosine | Matches reference |
| `optimizer` | paged_adamw_32bit | Matches reference |

### Next Steps
- **Phase 4.5**: Run `Phase4_5_Stage2_QLoRA_ORPO.ipynb` to align Stage 1 model using `ORPOTrainer` on UltraFeedback, HH-RLHF, and HaluEval.
- **Before Phase 4.5**: Merge Stage 1 adapter into the base model so ORPO has a clean starting point.